# M01-01 — Sesión Spark y primer DataFrame

Referencia de validación. El alumno trabaja en `notebooks/alumno/M01-01-sesion-spark.ipynb`.


## Celda 0 — localizar el repo


In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


## 1 — Runtime


In [ ]:
import shutil, subprocess, pyspark
print("pyspark", pyspark.__version__)
print(subprocess.check_output(["java", "-version"], text=True, stderr=subprocess.STDOUT).splitlines()[0])
print("java:", shutil.which("java"))
assert pyspark.__version__.startswith("3.5")


## 2 — Sesión


In [ ]:
spark = get_spark("novashop-m01")
print(spark)
assert spark.sparkContext.master.startswith("local")


## 3 — Pedidos en memoria


In [ ]:
from pyspark.sql import Row
pedidos = [
    Row(order_id="O90001", customer_id="C0001", status="paid", amount=49.90),
    Row(order_id="O90002", customer_id="C0002", status="paid", amount=12.50),
    Row(order_id="O90003", customer_id="C0003", status="cancelled", amount=80.00),
    Row(order_id="O90004", customer_id="C0001", status="paid", amount=23.10),
    Row(order_id="O90005", customer_id="C0004", status="pending", amount=5.00),
]
df = spark.createDataFrame(pedidos)
df.printSchema()
df.show()
assert df.count() == 5


## 4 — filter vs count


In [ ]:
paid = df.filter(df.status == "paid")
print("después del filter, Spark aún no ha contado nada")
print("paid count =", paid.count())
assert paid.count() == 3


## Comprueba + reto


In [ ]:
from pyspark.sql.functions import lit
print("spark.version", spark.version)
assert paid.count() == 3
df.withColumn("channel", lit("web")).select("order_id", "channel").show()
print("M01-01 OK")
